# Kerala helpdesk: does a Whisper initial_prompt help? (eval/asr_bias.py)

Transcribes the logged voice questions (`eval/asr_clips.csv`) with each prompt in `eval/asr_prompts.txt`, using the Space's
settings (own Whisper v2, CTranslate2 int8), and reports CER, service-word recall and **same answer** (the helpdesk routes the
transcript as it routes the reference). Then the same prompts on leakage-free corpus clips: **general WER** must not get worse.

**Setup**: Add-ons → Secrets → `HF_TOKEN` (read token; the usage log `sajilck/tts-demo-evidence-log` is private), attached to
this notebook. Add Input → Kaggle dataset `sajilck/mal-whisper-v2-input` (for `own_corpus_test_clean.csv`). Internet on. CPU is enough.
Run with **Save Version → Save & Run All**; download `asr_bias_out.zip` from Output.

The audio is visitors' speech: this notebook keeps it private and never sends it to an API. Do not make this notebook public.

In [ ]:
# ---- CONFIG ----
REPO_URL   = "https://github.com/cksajil/kerala-voice-helpdesk"
LOG_DS     = "sajilck/tts-demo-evidence-log"     # private usage log with audio/<day>/<id>_in.wav
WER_DS     = "sajilck/malayalam-asr-corpus"      # general WER check: test split, leakage-free rows only
PER_SOURCE = 40                                  # clips per source for the general check (~200 in all)
CANON      = "1"                                 # as on the Space

In [ ]:
import os, subprocess, sys, glob, shutil, csv, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.chdir("/kaggle/working")
shutil.rmtree("repo", ignore_errors=True)        # always a fresh clone: never run old code
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper", "ctranslate2", "rank_bm25",
                "langgraph", "feedparser", "librosa"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
print(open("eval/asr_prompts.txt", encoding="utf-8").read())

In [ ]:
# Fetch only the logged clips listed in eval/asr_clips.csv (found by id, wherever they sit in the dataset)
from huggingface_hub import HfApi, hf_hub_download
ids = {r["id"] for r in csv.DictReader(open("eval/asr_clips.csv", encoding="utf-8"))}
files = HfApi().list_repo_files(LOG_DS, repo_type="dataset")
want = [f for f in files if f.endswith("_in.wav") and f.split("/")[-1].split("_")[0] in ids]
print(len(files), "files in the log;", len(want), "of", len(ids), "clips found")
if not want:
    print("examples of audio paths:", [f for f in files if f.endswith(".wav")][:5])
for f in want:
    hf_hub_download(LOG_DS, f, repo_type="dataset", local_dir="eval/asr_audio")
print(len(glob.glob("eval/asr_audio/**/*_in.wav", recursive=True)), "wav files ready")

In [ ]:
clean = glob.glob("/kaggle/input/**/own_corpus_test_clean.csv", recursive=True)
print("clean csv:", clean[:1] or "not attached: the general check will use the whole test split (has leaked rows)")
args = ["--show", "6", "--wer-dataset", WER_DS, "--wer-per-source", str(PER_SOURCE)] + (["--wer-clean-csv", clean[0]] if clean else [])
os.environ["HELPDESK_CANON"] = CANON
!python -u -W ignore -m eval.asr_bias --from-log
!python -u -W ignore -m eval.asr_bias {" ".join(args)}

In [ ]:
out = pathlib.Path("/kaggle/working/asr_bias_out"); shutil.rmtree(out, ignore_errors=True); out.mkdir()
shutil.copy("eval/asr_bias_report.json", out)        # transcripts and scores only; no audio leaves the notebook
shutil.make_archive("/kaggle/working/asr_bias_out", "zip", out)
shutil.rmtree("eval/asr_audio", ignore_errors=True)   # keep visitors' audio out of the saved Output
print("download asr_bias_out.zip from Output")